# Using Plotly

## Notes
- I used `pd.merge` to keep all datetimes, not only the common ones. This is more reasonable for plotting, as we can now see where we're missing which data.
- How to interact with the plots: 
  - hover vertically to see bike count, temperature, and rain of a specific datetime
  - use rangeslider to zoom in
  - double click to zoom out completely 
  - click on variable to show/hide the line in the plot
- Counter sites for Heidelberg and Mannheim are arbitrarily picked, we should think about which sites are useful (or combine various) 

In [1]:
def plot_data(city, counter_name):
    """Fetch data for a given city and counter site, merge bike counts with weather data, and create an interactive plot.

    Args:
        city (str): Name of the city, written as in weather file names.
        counter_name (str): Counter site of interest.
    """    
    import pandas as pd
    import plotly.express as px
    from get_table import get_table
    
    # get hourly weather data for the specified city (2024-11-01 to 2025-10-31)
    weather_hourly = pd.read_csv(f'weather_data/hourly/{city.lower()}_weather_2024-11-01_2025-10-31.csv')

    # get hourly bike count data for the specified city (2024-11-01 to 2025-10-31)
    bike_hourly = get_table('eco-counter/all_cities', 2024, 11, 1, 2025, 10, 31)
    bike_hourly = bike_hourly[bike_hourly['counter_site'] == counter_name]

    # Add a new column datetime in both tables (for same name)
    weather_hourly['datetime'] = pd.to_datetime(weather_hourly['time'])
    bike_hourly['datetime'] = pd.to_datetime(bike_hourly['iso_timestamp'])

    # Merge both tables on datetime to get all data in one table
    df_common = pd.merge(
        bike_hourly[['datetime', 'channels_all']],
        weather_hourly[['datetime', 'prcp', 'temp']],
        on='datetime',
        how='outer'
    ).sort_values('datetime')

    # Rename columns for clarity
    df_common.rename(columns={'channels_all': 'bike', 'prcp': 'rain', 'temp': 'temp'}, inplace=True)

    df_common[['bike', 'rain', 'temp']] = df_common[['bike', 'rain', 'temp']].apply(
        pd.to_numeric, errors='coerce'
    )
    
    fig = px.line(df_common, x="datetime", y=df_common.columns,
                  category_orders={"variable": ["bike", "rain", "temp"]},
                  color_discrete_map={    # Custom colors
                    "bike": px.colors.qualitative.Set3[0],
                    "temp": px.colors.qualitative.Set3[3],
                    "rain": px.colors.qualitative.Set3[4],
                    },
                  title=f"Bike Counts and Weather Data for {city}, {counter_name}")

    fig.update_layout(hovermode="x unified")        # On hover, show all values depending on x axis together
    fig.update_traces(hovertemplate=None)           # Compact hover info

    fig.update_xaxes(rangeslider_visible=True)      # Add range slider

    fig.show()


In [2]:
plot_data("Tuebingen", "Fuß- & Radtunnel Südportal - Derendinger Allee")

In [3]:
plot_data("Mannheim", "Fernmeldeturm.")

In [4]:
plot_data("Heidelberg", "Mannheimer Straße")

Interesting to see here: the spike on August 22 – August 24 is probably the "Street Food & Music Festival Heidelberg" which took place on the "Messplatz", which is very close to the counter at Mannheimer Straße. 

## Use graph opjects to have multiple y-axes

In [5]:
def plot_data_overlay(city, counter_name):
    """Fetch data for a given city and counter site, merge bike counts with weather data, and create an interactive plot.

    Args:
        city (str): Name of the city, written as in weather file names.
        counter_name (str): Counter site of interest.
    """    
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    from get_table import get_table
    
    # get hourly weather data for the specified city (2024-11-01 to 2025-10-31)
    weather_hourly = pd.read_csv(f'weather_data/hourly/{city.lower()}_weather_2024-11-01_2025-10-31.csv')

    # get hourly bike count data for the specified city (2024-11-01 to 2025-10-31)
    bike_hourly = get_table('eco-counter/all_cities', 2024, 11, 1, 2025, 10, 31)
    bike_hourly = bike_hourly[bike_hourly['counter_site'] == counter_name]

    # Add a new column datetime in both tables (for same name)
    weather_hourly['datetime'] = pd.to_datetime(weather_hourly['time'])
    bike_hourly['datetime'] = pd.to_datetime(bike_hourly['iso_timestamp'])

    # Merge both tables on datetime to get all data in one table
    df_common = pd.merge(
        bike_hourly[['datetime', 'channels_all']],
        weather_hourly[['datetime', 'prcp', 'temp']],
        on='datetime',
        how='outer'
    ).sort_values('datetime')

    # Rename columns for clarity
    df_common.rename(columns={'channels_all': 'bike', 'prcp': 'rain', 'temp': 'temp'}, inplace=True)

    df_common[['bike', 'rain', 'temp']] = df_common[['bike', 'rain', 'temp']].apply(
        pd.to_numeric, errors='coerce'
    )
    
    fig = go.Figure()
    
    # Bike trace
    fig.add_trace(go.Scatter(
        x=df_common['datetime'],
        y=df_common['bike'],
        name='Bike Count',
        line=dict(color=px.colors.qualitative.Set3[0]),
        yaxis='y1'
    ))
    # Temperature trace
    fig.add_trace(go.Scatter(
        x=df_common['datetime'],
        y=df_common['temp'],
        name='Temperature',
        line=dict(color=px.colors.qualitative.Set3[3]),
        yaxis='y2'
    ))
    # Rain trace
    fig.add_trace(go.Scatter(
        x=df_common['datetime'],
        y=df_common['rain'],
        name='Rainfall',
        line=dict(color=px.colors.qualitative.Set3[4]),
        yaxis='y3'
    ))
    
    
    # Layout for multiple y-axes
    fig.update_layout(
        title=f"Bike Counts and Weather Data for {city}, {counter_name}",
        
        legend={
            "y": 1.5,
        },
        
        xaxis=dict(
            title='Datetime',
            rangeslider=dict(visible=True),
            #domain=[0.25, 0.75]
        ),
        
        yaxis=dict(
            title=dict(
                text="Bike Count [#]",
                font=dict(
                    color=px.colors.qualitative.Set3[0]
                ),
            ),
            side='left',
            showgrid=False
        ),
        yaxis2=dict(
            title=dict(
                text="Temperature [°C]",
                font=dict(
                    color=px.colors.qualitative.Set3[3]
                ),
            ),
            anchor="x",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
        yaxis3=dict(
            title=dict(
                text="Rainfall [mm]",
                font=dict(
                    color=px.colors.qualitative.Set3[4]
                ),
            ),
            anchor="free",
            overlaying="y",
            side="right",
            autoshift=True,
            showgrid=False,
            shift=50,
        ),
        hovermode='x unified',
    )

    fig.show()


In [6]:
plot_data_overlay("Tuebingen", "Fuß- & Radtunnel Südportal - Derendinger Allee")

In [7]:
plot_data_overlay("Mannheim", "Fernmeldeturm.")

In [8]:
plot_data_overlay("Heidelberg", "Mannheimer Straße")